# Video & 3D Generation- Turning Diffusion Up a Dimension

**Module 6 · Lesson 6.6**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- How Video Generation Works - Images + Time
- The Architecture - Latent Video Diffusion + a Diffusion Transformer
- Generating Video via API - Text-to-Video & Image-to-Video
- 3D from Photos - Gaussian Splatting
- Text-to-3D & Image-to-3D - Game-Ready Assets
- The 2026 Landscape, Economics & Responsible Use

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install torch -q

# Reproducibility - the lesson uses seeded random values throughout

## The whole game: one machine, more dimensions

> 🎬 **Analogy**
>
> **You already know how to turn noise into an image (Lesson 6.1) and how to cut an image into patch tokens a transformer can read (Lesson 6.4).** Video generation adds one axis - **time** - and asks the model to make 24+ images per second that all agree with each other. 3D generation adds a different axis - **space** - and asks the model to build a scene you can view from any angle. Same denoising, same transformer attention; one more dimension to be consistent across.
> Everything in this lesson is that one idea: **generation across time and space**. Text-to-video, image-to-video, photos-to-3D, text-to-3D - all the same diffusion-and-attention machinery, now generating instead of understanding.
> **Where the analogy breaks down:** adding a dimension is not free. A ten-second clip has hundreds of times more data than one image, and every frame must stay consistent with its neighbours - so video models are expensive, clips are short, and the hardest bugs are drift and flicker, not a single bad frame. That cost is the thread running through the whole lesson.

This is the **climax** of Module 6: the payoff of everything before it. It builds directly on Lesson 6.1 (diffusion / denoising), Lesson 6.4 (the ViT that turns images into patch tokens, which becomes the video model's Diffusion Transformer), and Lesson 6.5 (multimodal - one model, many modalities). It forward-hooks Lesson 6.7 (voice, the last modality), Module 11 (agents that drive these tools), and Module 15 (deepfakes, provenance, and the responsible use of generative video).

- **Explain** why video generation is hard (temporal consistency; a 10s clip is ~240x the data of one image) and how latent video diffusion plus a Diffusion Transformer over spacetime patches solve it - reusing diffusion (6.1) and the ViT (6.4).

- **Apply** a modern video API for text-to-video and image-to-video using the async submit-then-poll pattern, and know when to self-host an open model (Wan / HunyuanVideo / LTX-Video).

- **Explain** how 3D Gaussian Splatting turns photos into a renderable 3D scene, and why explicit Gaussians overtook NeRF's implicit field for production.

- **Analyze** the 2026 landscape and its economics - closed vs open, native audio, the cost-per-output lesson behind Sora's shutdown - plus the responsible-use stack (provenance, watermarking, deepfakes) that Module 15 develops.

## Warm-up: 60 seconds of recall

Tap each card to check yourself. Each one is load-bearing for today.

---

## 🎯 Concept 1: How Video Generation Works - Images + Time

### How Video Generation Works - Images + Time

A video is 24+ images per second with one hard rule: every frame must agree with the last one.

**A flipbook.** Flip drawings fast and you see motion. But if each page were drawn by a different artist with no coordination, the character would change size and the background would jump every page. A video model is one coordinated artist who draws every page remembering the last one - so the motion is smooth and the character stays itself.

The gain: a video = images + a consistency constraint across time. The frames must agree, and there are a lot of them.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

**📝 `video_is_expensive.py`** - *PyTorch*

In [ ]:
# A video is just images stacked on a time axis. Let's feel the data cost.
import torch

image    = torch.randn(3, 512, 512)          # one RGB frame: (channels, H, W)
video_1s = torch.randn(24, 3, 512, 512)      # 1 second at 24 fps: (frames, C, H, W)
video_10s = torch.randn(240, 3, 512, 512)    # 10 seconds: 240 frames

def millions(t):
    return t.numel() / 1e6              # total values, in millions

print(f"1 image:      {millions(image):.2f} M values")
print(f"1 sec video:  {millions(video_1s):.2f} M values  ({video_1s.shape[0]}x an image)")
print(f"10 sec video: {millions(video_10s):.2f} M values  ({video_10s.shape[0]}x an image)")
# Output: 1 image:      0.79 M values
# Output: 1 sec video:  18.87 M values  (24x an image)
# Output: 10 sec video: 188.74 M values  (240x an image)

- A video tensor just adds a **frames** axis in front of the image's `(C, H, W)`. Ten seconds at 24 fps is 240 frames.

- `numel()` counts every value in the tensor. Ten seconds of video is **240x** the values of a single image, at the same resolution - and diffusion has to denoise all of them, many times.

- That 240x is the whole reason video is expensive, clips are short (most models cap at 5-15s), and duration is a real cost dial, not a free parameter. Step 2 shows the trick that makes it tractable: compress first.

#### 💡 What just happened

⚡ What just happened?
  A video is images plus a hard **temporal-consistency** constraint - every frame must agree with its neighbours, or you get flicker and drift. And it is **big**: a 10-second clip is ~240x the data of one image. Those two facts - consistency and cost - drive every design choice in the rest of the lesson.

---

## 🎯 Concept 2: The Architecture - Latent Video Diffusion + a Diffusion Transformer

### The Architecture - Latent Video Diffusion + a Diffusion Transformer

Compress the clip into a latent, then let a transformer denoise "spacetime patches" that attend across time.

**Zip the film reel, then read it in chunks.** First squeeze the whole clip into a compact "zip" (the latent) so there is far less to work on. Then hand a transformer little 3D chunks - a small square of the picture across a few frames at once - so when it cleans up one chunk it is looking at what happened just before and after. That is what keeps the motion glued together.

The gain: 3D VAE (compress) + Diffusion Transformer over spacetime patches (attend across time). It is 6.1's denoising and 6.4's patch-tokens, extended one axis.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

**📝 `latent_and_patches.py`** - *PyTorch*

In [ ]:
import torch

video_10s = torch.randn(240, 3, 512, 512)   # raw: (frames, C, H, W)

# A 3D VAE compresses ~8x per spatial side AND groups frames in time.
latent = torch.randn(60, 4, 64, 64)         # (time-groups, ch, H/8, W/8)

# A Diffusion Transformer cuts the LATENT into spacetime patches -
# like ViT patches from 6.4, but 3D (a bit of space across a few frames).
patch_t, patch_s = 2, 2                      # 2 time-groups x 2x2 spatial per patch
n_patches = (60 // patch_t) * (64 // patch_s) * (64 // patch_s)

print(f"raw values:        {video_10s.numel():,}")
print(f"latent values:     {latent.numel():,}  ({video_10s.numel() // latent.numel()}x smaller)")
print(f"spacetime patches: {n_patches:,}  <- the tokens the transformer denoises")
# Output: raw values:        188,743,680
# Output: latent values:     983,040  (192x smaller)
# Output: spacetime patches: 30,720  <- the tokens the transformer denoises

- **3D VAE (compress):** encode the raw clip into a small latent - ~192x fewer values here - so denoising is affordable. This is the same latent-diffusion idea as Stable Diffusion (6.1-6.2), now compressing time as well as space.

- **Spacetime patches (tokenize):** the Diffusion Transformer cuts the latent into 3D chunks and treats each as a token - exactly 6.4's ViT patches, extended with a time slice. Sora-class models call these "spacetime patches".

- **Temporal attention (stay consistent):** the transformer's attention lets each patch attend to the same region in neighbouring frames, so a thrown ball keeps its arc and a face stays itself. That cross-frame attention is what a plain image model lacks.

- **Native audio (2026):** the newest models generate synchronized dialogue and sound jointly with the video, not as a separate pass - Veo leads here (48kHz synchronized dialogue), with Kling offering multilingual lip-sync.

#### 💡 What just happened

⚡ What just happened?
  The winning recipe is **latent video diffusion + a Diffusion Transformer**: compress the clip with a **3D VAE**, denoise **spacetime patches** with a transformer, and use **temporal attention** to keep frames consistent - then decode back to video. It is 6.1's diffusion and 6.4's ViT, one dimension up. An alternative, generating frames one-after-another (autoregressive), is simpler but drifts over long clips.

---

## 🎯 Concept 3: Generating Video via API - Text-to-Video & Image-to-Video

### Generating Video via API - Text-to-Video & Image-to-Video

Submit a job, poll for the result. The prompt and the starting image are your control surface.

**Ordering food at a busy kitchen counter.** You do not stand at the window until the dish is cooked. You place the order, get a ticket number, and check back (or they buzz you) when it is ready. A video API is the same: submit the job, hold the id, poll until the clip is plated.

The gain: video generation is a job you submit and poll, not a call that blocks. And the prompt plus a starting image steer the result, just like Module 5.

**📝 `generate_video.py Runway`** - *API*

In [ ]:
# pip install runwayml   -  image-to-video via a hosted API (async: submit, then poll)
import os, time
from runwayml import RunwayML

client = RunwayML(api_key=os.environ["RUNWAYML_API_SECRET"])

# 1) SUBMIT a job - you get a task id back, NOT the video.
task = client.image_to_video.create(
    model="gen4_turbo",
    prompt_image="https://example.com/product.jpg",   # a still to animate
    prompt_text="Camera slowly orbits the product, soft studio light, smooth motion.",
    duration=5,                                         # seconds
    ratio="1280:720",
)

# 2) POLL until it is ready (in production, prefer a webhook over a loop).
while True:
    task = client.tasks.retrieve(task.id)
    if task.status in ("SUCCEEDED", "FAILED"):
        break
    time.sleep(5)

print(task.status)
print(task.output[0] if task.status == "SUCCEEDED" else task.failure)
# Output: SUCCEEDED
# Output: https://runway-outputs.../gen4_turbo_orbit.mp4

- **Async is the whole shape:** `create()` returns a task id immediately; you `retrieve()` until the status is terminal. A synchronous "give me the MP4 now" call would time out - this is the pattern for every video API (Runway, Veo, Kling).

- **The URL expires:** the result is a *signed, time-limited* URL (hours to days), not the video bytes - download the MP4 and re-host it to your own object storage immediately; never persist the vendor URL as your asset of record, or it will 404 later.

- **What it costs:** the 5-second `gen4_turbo` call shown is roughly $0.25 per clip as of mid-2026 (Kling runs cheaper at scale, near $0.50 for a longer clip) - always check current pricing before wiring a paid API into a request path.

- **Image-to-video vs text-to-video:** pass a `prompt_image` to animate a still (product demos, "make this photo move"); drop it for pure text-to-video. The `prompt_text` steers motion and camera, exactly like Module 5 prompting.

- **Hosted alternatives:** **Veo 3.1** (via Vertex AI / Gemini, native synchronized audio, up to 4K) and **Kling 3.0** (cheap at scale, multilingual lip-sync) use the same submit-and-poll shape with different SDKs.

- **Self-hosted open:** to keep data private or cut per-clip cost, run an open model locally - `diffusers` loads **Wan** (Apache-2.0, the 1.3B fits ~8 GB VRAM), **HunyuanVideo**, or **LTX-Video**. Same idea, your GPU.

- **Model IDs move:** `gen4_turbo` is the fast tier of Runway's Gen-4 line (the newer flagship, Gen-4.5, appears in Step 6); model and version names are the mid-2026 generation - check the provider's current model list, and downscale duration/resolution to control cost.

#### 💡 What just happened

⚡ What just happened?
  Generating video is one shape: **submit a job, poll for a result URL**. Pass a starting image for image-to-video or just text for text-to-video; the prompt steers motion and camera. Hosted APIs (Veo, Kling, Runway) trade money for quality and zero ops; self-hosted open models (Wan, HunyuanVideo, LTX) trade GPU setup for privacy and per-clip cost.

---

## 🎯 Concept 4: 3D from Photos - Gaussian Splatting

### 3D from Photos - Gaussian Splatting

Take 40 phone photos of a scene and get a 3D model you can fly through from any angle.

**A swarm of tiny coloured cotton balls.** Imagine filling the shape of a statue with millions of soft, semi-transparent coloured puffs. From a distance, overlapping puffs blend into a solid, photorealistic surface - and because they float in 3D, you can walk around and see them from any side. That swarm is a Gaussian Splat: each puff is one 3D Gaussian - called a "Gaussian" because it is densest in the middle and fades softly to nothing at its edges (the bell-curve shape), which is exactly why overlapping puffs blend into a smooth surface instead of looking like hard dots.

The gain: a scene = millions of explicit 3D blobs (position, size, colour, opacity), optimized to match your photos. Render any viewpoint by splatting them - fast.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

**📝 `gaussian_splatting.py`** - *nerfstudio*

In [ ]:
# What ONE 3D Gaussian holds - a scene is millions of these.
from dataclasses import dataclass

@dataclass
class Gaussian3D:
    position: tuple   # (x, y, z) - where the blob sits in space
    scale:    tuple   # (sx, sy, sz) - its size and shape (the covariance)
    color:    tuple   # (r, g, b) - view-dependent, stored as spherical harmonics
    opacity:  float   # 0..1 - how solid vs see-through it is

blob = Gaussian3D(position=(0.0, 0.1, 2.3), scale=(0.02, 0.02, 0.05),
                  color=(0.8, 0.2, 0.2), opacity=0.9)
print(blob)
print("a real indoor room ~ 1-2M Gaussians, rendered at 100+ FPS")
# Output: Gaussian3D(position=(0.0, 0.1, 2.3), scale=(0.02, 0.02, 0.05), color=(0.8, 0.2, 0.2), opacity=0.9)
# Output: a real indoor room ~ 1-2M Gaussians, rendered at 100+ FPS

# Run it on your own photos with nerfstudio (open, well-documented):
#   pip install nerfstudio
#   ns-process-data images --data ./photos --output-dir ./scene   # SfM camera poses (COLMAP)
#   ns-train splatfacto --data ./scene                            # optimize the Gaussians (minutes on a GPU)
#   ns-viewer --load-config ./outputs/.../config.yml              # fly through the result
# Prefer a phone app? Luma or Polycam do capture -> splat in the cloud.

- Each Gaussian is **explicit**: a position, a size/shape (covariance), a colour (stored as spherical harmonics so it can change with viewing angle), and an opacity. Millions of them, overlapping, render a photorealistic surface.

- The pipeline is **capture -> SfM -> optimize -> splat**: `ns-process-data` runs Structure from Motion (COLMAP) to recover camera poses; `ns-train splatfacto` optimizes the Gaussians to match your photos in minutes; then you render any viewpoint at 100+ FPS.

- Because the blobs are explicit points, you can **edit** them (crop, move, recolour) and rasterize them fast - the two properties that made Gaussian Splatting the production default.

#### 💡 What just happened

⚡ What just happened?
  Gaussian Splatting turns a set of photos into a 3D scene made of **millions of explicit coloured blobs**, optimized to reproduce your photos, then rendered from any angle at 100+ FPS. The pipeline is capture, recover camera poses (SfM), optimize the Gaussians, and splat. It overtook **NeRF** - which stores the scene as an implicit neural network queried by slow ray-marching - because explicit blobs are fast to render and easy to edit.

---

## 🎯 Concept 5: Text-to-3D & Image-to-3D - Game-Ready Assets

### Text-to-3D & Image-to-3D - Game-Ready Assets

From a prompt to a mesh you can drop into Blender, Unity, or Unreal - in seconds, with cleanup.

**Sculpting by walking around the clay.** A sculptor checks the piece from every side and fixes whatever looks wrong from that angle, over and over, until it looks right from all sides. Text-to-3D does the same: it keeps asking a 2D image model "does this look right from here?" and adjusting the 3D shape - so it never needs 3D training data, just a good 2D critic.

The gain: text-to-3D distills a 2D diffusion model into a 3D shape (SDS). In 2026, feed-forward image-to-3D does it in seconds - and image-to-3D beats text-to-3D.

**📝 `image_to_3d.py`** - *Hunyuan3D*

In [ ]:
# The reliable 2026 pipeline: text -> one clean 2D image -> feed-forward image-to-3D.
# Step A: make/choose a single clean reference image (SD/Flux from Lesson 6.2).
# Step B: convert it to 3D with an open feed-forward model (seconds, not hours).
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline   # shape (open)
from hy3dgen.texgen import Hunyuan3DPaintPipeline                # texture (open)
from PIL import Image

img = Image.open("chair_reference.png")
shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2")
mesh = shape(image=img)[0]                 # image -> untextured mesh (shape only)
paint = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2")
mesh = paint(mesh, image=img)              # add PBR texture from the reference image
mesh.export("chair.glb")                   # glTF: import into Blender / Unity / Unreal
print("saved chair.glb  -  expect 15-30 min of cleanup: topology, UVs, the occluded back")
# Output: saved chair.glb  -  expect 15-30 min of cleanup: topology, UVs, the occluded back

- **Why image-to-3D, not text-to-3D:** a single clean reference image pins down what the object looks like, so the 3D model has far less to invent. Going text -> image (SD/Flux) -> 3D is consistently more reliable than asking for 3D straight from text.

- **Feed-forward, not optimization:** models like **Hunyuan3D** and **TRELLIS** produce an asset in seconds. TRELLIS can output Gaussian splats or meshes; Hunyuan3D outputs meshes with PBR materials (`.glb`, ready for a game engine). SDS-style optimization (DreamFusion) still exists but takes hours per object.

- **Cleanup is real:** expect 80-95% shape accuracy on front-facing surfaces and manual fixes on the occluded back, topology, and UVs. Hosted tools (Meshy, Tripo, Rodin) trade the local setup for a subscription. The exact pipeline call is on each model card - APIs move quickly.

#### 💡 What just happened

⚡ What just happened?
  Text-to-3D works by **distilling a 2D diffusion model into a 3D shape** (Score Distillation Sampling, DreamFusion) - no 3D training data needed. In 2026, production moved to **feed-forward image-to-3D** (TRELLIS, Hunyuan3D, Meshy) that outputs an asset in seconds, and the reliable pipeline is text -> image -> 3D -> artist polish. Image-to-3D beats text-to-3D because a clean reference removes the guessing.

---

## 🎯 Concept 6: The 2026 Landscape, Economics & Responsible Use

### The 2026 Landscape, Economics & Responsible Use

Closed vs open, native audio, the cost-per-output lesson from Sora, and the deepfake responsibility.

**The fastest race car that costs more per lap than the prize money is not the winner.** Raw capability is only half the equation - if every "lap" (every generated clip) loses money, the team folds. That is why lighter and open models, which cost far less per clip, keep winning ground even when a flashier model exists.

The gain: in production, cost-per-output and privacy decide as much as quality. Native audio and open self-hosting are the 2026 differentiators.

| Model | Type | Strength (as of mid-2026) | Reach for it when |
|---|---|---|---|
| Veo 3.1(Google) | Closed, native | Best all-rounder; up to 4K + native synchronized audio | You want top quality with sound, least ops |
| Runway Gen-4.5 | Closed | Best control surface (motion brushes, consistency) | Film/production work, precise camera control |
| Kling 3.0(Kuaishou) | Closed | Cheap at scale; multilingual lip-sync | High-volume short-form, tight budget |
| Wan(Alibaba) | Open (Apache-2.0) | 1.3B runs in ~8 GB VRAM; T2V + I2V | Privacy / cost / self-hosting |
| HunyuanVideo / LTX-Video | Open | Full-attention DiT / fastest open model | Local generation, custom pipelines |
| Sora 2(OpenAI) | Closed | Retired: app Apr 2026, API sunset Sep 2026 | The cost-per-output cautionary tale |

**📝 `choose_model.py`** - *Decision*

In [ ]:
# Pick a video model by the constraint that binds - not by benchmark alone.
def choose_video_model(need: str) -> str:
    table = {
        "top_quality":    "Veo 3.1 (native audio, up to 4K) or Runway Gen-4.5 (best control)",
        "cheap_at_scale": "Kling 3.0 (roughly $0.50/clip, fast, multilingual lip-sync)",
        "privacy":       "self-host Wan (Apache-2.0, 1.3B runs in ~8 GB VRAM)",
        "native_audio":  "Veo 3.1 native synchronized audio; Kling 3.0 lip-sync",
    }
    return table.get(need, "start hosted; move to open self-host when cost or privacy forces it")

for need in ["top_quality", "cheap_at_scale", "privacy", "native_audio"]:
    print(f"{need:14} -> {choose_video_model(need)}")
# Output: top_quality    -> Veo 3.1 (native audio, up to 4K) or Runway Gen-4.5 (best control)
# Output: cheap_at_scale -> Kling 3.0 (roughly $0.50/clip, fast, multilingual lip-sync)
# Output: privacy        -> self-host Wan (Apache-2.0, 1.3B runs in ~8 GB VRAM)
# Output: native_audio   -> Veo 3.1 native synchronized audio; Kling 3.0 lip-sync

- **Constraint, not benchmark:** the right model is chosen by what binds - quality, cost at volume, privacy, or native audio - which is exactly the lesson Sora taught by shutting down despite its quality.

- **Closed vs open:** closed APIs (Veo, Runway, Kling) win on quality and zero ops; open self-host (Wan, HunyuanVideo, LTX) wins on privacy and per-clip cost. Native synchronized audio is the 2026 leap and currently a closed-model strength.

- **The numbers move:** pricing and version names are mid-2026 and change monthly - treat this as a decision procedure, not a fixed leaderboard, and re-check current models and prices.

> 📦 **Info**
>
> ⚠️ The responsibility that ships with generative video
> Photorealistic video and voice make **deepfakes** cheap, so provenance is now part of the build, not an afterthought. Two layers you should know by name: **C2PA Content Credentials** (cryptographically signed "who made this and how" metadata) and invisible **watermarking** like Google **SynthID**, which survives cropping and compression. In production you attach provenance *at generation time*, not after. The deep treatment - bias, consent, disclosure, and the regulatory landscape - is **Module 15 (Ethics & Responsible AI)**; here, just internalize that shipping generative video means shipping provenance with it.

#### 💡 What just happened

⚡ What just happened?
  The 2026 landscape splits into **closed leaders** (Veo 3.1 all-round + native audio, Runway Gen-4.5 control, Kling 3.0 cheap at scale) and **open leaders** (Wan, HunyuanVideo, LTX-Video) you can self-host. **Sora's retirement** is the cost-per-output lesson: capability is worthless if the unit economics do not close. And generative video ships with a responsibility - **provenance and watermarking** (C2PA, SynthID) - that Module 15 develops.

## Synthesis: the same machine, one more dimension

### The complete picture, in one breath

Video and 3D generation are **diffusion (6.1) plus the transformer (6.4), turned up a dimension**. A video is images plus a temporal-consistency constraint; you make it affordable with a **3D VAE** and consistent with **temporal attention** in a **Diffusion Transformer** over spacetime patches, and you generate it through an API by **submitting a job and polling**. 3D from photos is **Gaussian Splatting** - millions of explicit blobs optimized to match your shots, rendered from any angle, which beat NeRF on speed and editability. 3D from a prompt is **image-to-3D** (feed-forward, image beats text, expect cleanup). And the 2026 reality is economic and ethical: **cost-per-output** decides as much as quality (the Sora lesson), open models rival closed for privacy and cost, and generative video ships with **provenance**. One machine; time and space are just two more axes to be consistent across.

> ℹ️ **Info**
>
> Where this goes next
> - **Lesson 6.7 (Voice AI: STT & TTS)** closes the module with the last modality - speech in and speech out - the audio partner to this lesson's video.
> - Agents that *drive* these generation tools (plan a shot list, call the video and 3D APIs, assemble a result) are covered in Module 11.
> - The responsible-use stack - deepfakes, consent, provenance (C2PA / SynthID), disclosure, and regulation - gets its full treatment in Module 15.

- Ho et al., *Video Diffusion Models* (2022) - [arxiv.org/abs/2204.03458](https://arxiv.org/abs/2204.03458) (extending diffusion across time)

- Blattmann et al., *Stable Video Diffusion* (2023) - [arxiv.org/abs/2311.15127](https://arxiv.org/abs/2311.15127) (latent video diffusion)

- Peebles & Xie, *Scalable Diffusion Models with Transformers (DiT)* (2023) - [arxiv.org/abs/2212.09748](https://arxiv.org/abs/2212.09748) (the Diffusion Transformer, from ViT of 6.4)

- Kerbl et al., *3D Gaussian Splatting for Real-Time Radiance Field Rendering* (2023) - [arxiv.org/abs/2308.04079](https://arxiv.org/abs/2308.04079)

- Mildenhall et al., *NeRF* (2020) - [arxiv.org/abs/2003.08934](https://arxiv.org/abs/2003.08934) (the implicit field GS overtook)

- Poole et al., *DreamFusion: Text-to-3D using 2D Diffusion* (2022) - [arxiv.org/abs/2209.14988](https://arxiv.org/abs/2209.14988) (Score Distillation Sampling)

- Xiang et al., *Structured 3D Latents (TRELLIS)* (2024) - [arxiv.org/abs/2412.01506](https://arxiv.org/abs/2412.01506) · Runway API - [docs.dev.runwayml.com](https://docs.dev.runwayml.com/) · OpenAI Sora discontinuation - [help.openai.com](https://help.openai.com/en/articles/20001152-what-to-know-about-the-sora-discontinuation)

## Recap

> ✅ **Info**
>
> What we covered
> - **Why video is hard** - images plus a temporal-consistency constraint; a 10s clip is ~240x an image.
> - **The architecture** - 3D VAE (compress) + Diffusion Transformer over spacetime patches + temporal attention; diffusion (6.1) and ViT (6.4), one dimension up.
> - **Generating video** - text-to-video / image-to-video via the async submit-then-poll pattern; hosted vs self-hosted open.
> - **3D from photos** - Gaussian Splatting (capture, SfM, optimize, splat); explicit blobs beat NeRF's implicit field on speed and editability.
> - **Text/image-to-3D** - SDS (DreamFusion) then feed-forward image-to-3D; image-to-3D beats text-to-3D; budget cleanup.
> - **2026 landscape** - closed (Veo / Runway / Kling) vs open (Wan / HunyuanVideo / LTX); cost-per-output (the Sora lesson); provenance and deepfakes.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Video & 3D Generation- Turning Diffusion Up a Dimension**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-6_6.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-6.6.html` - regenerate this notebook whenever the source HTML is updated.*
